In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report

print("✅ TensorFlow :", tf.__version__)
print("✅ Imports OK !")

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 5
LR         = 1e-4
TRAIN_DIR  = "train"
TEST_DIR   = "test"

print("📁 Vérification :")
print(f"   Train : {'✅' if os.path.exists(TRAIN_DIR) else '❌ INTROUVABLE'}")
print(f"   Test  : {'✅' if os.path.exists(TEST_DIR)  else '❌ INTROUVABLE'}")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', shuffle=True
)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical',
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes
CLASS_NAMES = list(train_generator.class_indices.keys())

print(f"\n📊 Classes     : {CLASS_NAMES}")
print(f"📊 Train       : {train_generator.samples} images")
print(f"📊 Validation  : {val_generator.samples} images")
print(f"📊 Test        : {test_generator.samples} images")

In [ ]:
def build_model(base_model, num_classes, name):
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=output, name=name)
    model.compile(
        optimizer=Adam(LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f"✅ {name:15s} | Params: {model.count_params():>10,}")
    return model

print("🏗️  Construction des modèles...\n")
model_vgg = build_model(
    VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3)),
    NUM_CLASSES, "VGG16"
)
model_resnet = build_model(
    ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3)),
    NUM_CLASSES, "ResNet50"
)
model_mobile = build_model(
    MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3)),
    NUM_CLASSES, "MobileNetV2"
)
print("\n✅ Les 3 modèles sont prêts !")

In [ ]:
print("🔵 Entraînement VGG16...\n")
history_vgg = model_vgg.fit(
    train_generator, validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=3,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint('best_vgg16.keras', save_best_only=True, verbose=0)
    ]
)
with open('history_vgg.json', 'w') as f:
    json.dump(history_vgg.history, f)

print(f"\n✅ VGG16 terminé !")
print(f"   🎯 Meilleure Val Accuracy : {max(history_vgg.history['val_accuracy']):.2%}")

In [ ]:
# ============================================================
#  CELLULE 7 — ENTRAÎNEMENT RESNET50
# ============================================================

print("🟢 Entraînement ResNet50 en cours...\n")

history_resnet = model_resnet.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=7,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint('best_resnet50.keras',
                        save_best_only=True, verbose=0)
    ]
)

with open('history_resnet.json', 'w') as f:
    json.dump(history_resnet.history, f)

best_acc = max(history_resnet.history['val_accuracy'])
print(f"\n✅ ResNet50 terminé !")
print(f"   🎯 Meilleure Val Accuracy : {best_acc:.2%}")

In [ ]:
# ============================================================
#  CELLULE 8 — ENTRAÎNEMENT MOBILENETV2
# ============================================================

print("🟠 Entraînement MobileNetV2 en cours...\n")

history_mobile = model_mobile.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=3,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint('best_mobilenet.keras',
                        save_best_only=True, verbose=0)
    ]
)

with open('history_mobile.json', 'w') as f:
    json.dump(history_mobile.history, f)

best_acc = max(history_mobile.history['val_accuracy'])
print(f"\n✅ MobileNetV2 terminé !")
print(f"   🎯 Meilleure Val Accuracy : {best_acc:.2%}")

In [ ]:
# ============================================================
#  CELLULE 9 — COURBES D'APPRENTISSAGE
# ============================================================

histories = {
    'VGG16'      : history_vgg.history,
    'ResNet50'   : history_resnet.history,
    'MobileNetV2': history_mobile.history
}
COLORS = {
    'VGG16'      : '#3498db',
    'ResNet50'   : '#2ecc71',
    'MobileNetV2': '#e67e22'
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Courbes d'Apprentissage — VGG16 | ResNet50 | MobileNetV2",
             fontsize=16, fontweight='bold')

for idx, (name, h) in enumerate(histories.items()):
    c  = COLORS[name]
    ep = range(1, len(h['accuracy']) + 1)

    # Accuracy
    axes[0, idx].plot(ep, h['accuracy'],
                      label='Train', color=c,
                      linewidth=2, marker='o', markersize=5)
    axes[0, idx].plot(ep, h['val_accuracy'],
                      label='Validation', color=c,
                      linewidth=2, linestyle='--',
                      marker='s', markersize=5)
    axes[0, idx].set_title(f'{name} — Accuracy',
                            fontweight='bold', fontsize=12)
    axes[0, idx].set_xlabel('Epochs')
    axes[0, idx].set_ylabel('Accuracy')
    axes[0, idx].set_ylim(0, 1)
    axes[0, idx].legend()
    axes[0, idx].grid(True, alpha=0.3)
    axes[0, idx].fill_between(ep,
                               h['accuracy'],
                               h['val_accuracy'],
                               alpha=0.07, color=c)

    # Loss
    axes[1, idx].plot(ep, h['loss'],
                      label='Train', color=c,
                      linewidth=2, marker='o', markersize=5)
    axes[1, idx].plot(ep, h['val_loss'],
                      label='Validation', color=c,
                      linewidth=2, linestyle='--',
                      marker='s', markersize=5)
    axes[1, idx].set_title(f'{name} — Loss',
                            fontweight='bold', fontsize=12)
    axes[1, idx].set_xlabel('Epochs')
    axes[1, idx].set_ylabel('Loss')
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('courbes_apprentissage.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Courbes sauvegardées !")

In [ ]:
# ============================================================
#  CELLULE 10 — ÉVALUATION ET TABLEAU COMPARATIF
# ============================================================

all_models = {
    'VGG16'      : model_vgg,
    'ResNet50'   : model_resnet,
    'MobileNetV2': model_mobile
}

eval_results = {}
print("📊 Évaluation sur le Test Set...\n")

for name, model in all_models.items():
    test_generator.reset()
    loss, acc = model.evaluate(test_generator, verbose=0)
    eval_results[name] = {'accuracy': acc, 'loss': loss}
    bar = '█' * int(acc * 20) + '░' * (20 - int(acc * 20))
    print(f"  {name:15s} |{bar}| {acc:.2%}  Loss: {loss:.4f}")

# Tableau
data = []
for name, h in histories.items():
    data.append({
        'Modèle'      : name,
        'Train Acc'   : f"{max(h['accuracy']):.2%}",
        'Val Acc'     : f"{max(h['val_accuracy']):.2%}",
        'Test Acc'    : f"{eval_results[name]['accuracy']:.2%}",
        'Test Loss'   : f"{eval_results[name]['loss']:.4f}",
        'Best Epoch'  : h['val_accuracy'].index(max(h['val_accuracy']))+1,
        'Total Epochs': len(h['accuracy'])
    })

df   = pd.DataFrame(data)
best = max(eval_results, key=lambda x: eval_results[x]['accuracy'])

print("\n" + "="*72)
print("               🏆 TABLEAU COMPARATIF FINAL")
print("="*72)
print(df.to_string(index=False))
print("="*72)
print(f"\n  🥇 Meilleur modèle : {best}"
      f"  →  {eval_results[best]['accuracy']:.2%}")

In [ ]:
# ============================================================
#  CELLULE 11 — GRAPHIQUES COMPARATIFS FINAUX
# ============================================================

noms       = list(eval_results.keys())
accuracies = [eval_results[n]['accuracy'] for n in noms]
losses     = [eval_results[n]['loss']     for n in noms]
clrs       = [COLORS[n] for n in noms]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Résultats Finaux — Comparaison des 3 Modèles",
             fontsize=16, fontweight='bold')

# Accuracy
bars1 = axes[0].bar(noms, accuracies, color=clrs,
                    edgecolor='black', linewidth=0.8, width=0.5)
axes[0].set_title('Test Accuracy', fontweight='bold', fontsize=13)
axes[0].set_ylim(0, 1.15)
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f'{val:.2%}', ha='center',
                 fontweight='bold', fontsize=12)

# Loss
bars2 = axes[1].bar(noms, losses, color=clrs,
                    edgecolor='black', linewidth=0.8, width=0.5)
axes[1].set_title('Test Loss', fontweight='bold', fontsize=13)
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, losses):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center',
                 fontweight='bold', fontsize=12)

# Val Accuracy progression
for name, h in histories.items():
    ep = range(1, len(h['val_accuracy']) + 1)
    axes[2].plot(ep, h['val_accuracy'],
                 label=name, color=COLORS[name],
                 linewidth=2.5, marker='o', markersize=6)
axes[2].set_title('Val Accuracy par Epoch', fontweight='bold', fontsize=13)
axes[2].set_xlabel('Epochs')
axes[2].set_ylabel('Val Accuracy')
axes[2].set_ylim(0, 1)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparaison_finale.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphiques finaux sauvegardés !")

In [ ]:
# ============================================================
#  CELLULE 12 — MATRICES DE CONFUSION
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Matrices de Confusion — VGG16 | ResNet50 | MobileNetV2",
             fontsize=16, fontweight='bold')

for idx, (name, model) in enumerate(all_models.items()):
    test_generator.reset()
    preds  = model.predict(test_generator, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_generator.classes
    cm     = confusion_matrix(y_true, y_pred)

    sns.heatmap(cm,
                annot=True, fmt='d',
                cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES,
                ax=axes[idx],
                linewidths=0.5,
                annot_kws={"size": 11})
    axes[idx].set_title(
        f'{name}\nAcc: {eval_results[name]["accuracy"]:.2%}',
        fontweight='bold', fontsize=12
    )
    axes[idx].set_ylabel('Vraie Classe')
    axes[idx].set_xlabel('Classe Prédite')

plt.tight_layout()
plt.savefig('matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Matrices de confusion affichées !")

In [ ]:
# ============================================================
#  CELLULE 13 — RAPPORT DE CLASSIFICATION DÉTAILLÉ
# ============================================================

for name, model in all_models.items():
    test_generator.reset()
    preds  = model.predict(test_generator, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_generator.classes

    print(f"\n{'='*60}")
    print(f"   📋 RAPPORT DE CLASSIFICATION — {name}")
    print(f"{'='*60}")
    print(classification_report(
        y_true, y_pred,
        target_names=CLASS_NAMES,
        digits=4
    ))

In [ ]:
fig = plt.figure(figsize=(20, 18))
fig.suptitle("Résultats Complets — Deep Learning Médical",
             fontsize=18, fontweight='bold', y=0.98)

# ── Courbes Accuracy ──────────────────────────────────
for idx, (name, h) in enumerate(histories.items()):
    ax = fig.add_subplot(4, 3, idx+1)
    ep = range(1, len(h['accuracy'])+1)
    ax.plot(ep, h['accuracy'],     color=COLORS[name], linewidth=2, label='Train')
    ax.plot(ep, h['val_accuracy'], color=COLORS[name], linewidth=2,
            linestyle='--', label='Val')
    ax.set_title(f'{name} — Accuracy', fontweight='bold')
    ax.set_ylim(0, 1)
    ax.legend(); ax.grid(True, alpha=0.3)

# ── Courbes Loss ──────────────────────────────────────
for idx, (name, h) in enumerate(histories.items()):
    ax = fig.add_subplot(4, 3, idx+4)
    ep = range(1, len(h['loss'])+1)
    ax.plot(ep, h['loss'],     color=COLORS[name], linewidth=2, label='Train')
    ax.plot(ep, h['val_loss'], color=COLORS[name], linewidth=2,
            linestyle='--', label='Val')
    ax.set_title(f'{name} — Loss', fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

# ── Barres Test Accuracy ──────────────────────────────
ax = fig.add_subplot(4, 3, 7)
noms  = list(eval_results.keys())
accs  = [eval_results[n]['accuracy'] for n in noms]
clrs  = [COLORS[n] for n in noms]
bars  = ax.bar(noms, accs, color=clrs, edgecolor='black', width=0.5)
ax.set_title('Test Accuracy', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.02,
            f'{val:.2%}', ha='center', fontweight='bold')

# ── Barres Test Loss ──────────────────────────────────
ax = fig.add_subplot(4, 3, 8)
losses = [eval_results[n]['loss'] for n in noms]
bars2  = ax.bar(noms, losses, color=clrs, edgecolor='black', width=0.5)
ax.set_title('Test Loss', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, losses):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.005,
            f'{val:.4f}', ha='center', fontweight='bold')

# ── Val Accuracy progression ──────────────────────────
ax = fig.add_subplot(4, 3, 9)
for name, h in histories.items():
    ep = range(1, len(h['val_accuracy'])+1)
    ax.plot(ep, h['val_accuracy'], label=name,
            color=COLORS[name], linewidth=2.5, marker='o')
ax.set_title('Val Accuracy — Progression', fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(); ax.grid(True, alpha=0.3)

# ── Matrices de Confusion ─────────────────────────────
for idx, (name, model) in enumerate(all_models.items()):
    ax = fig.add_subplot(4, 3, idx+10)
    test_generator.reset()
    preds  = model.predict(test_generator, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_generator.classes
    cm     = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5)
    ax.set_title(f'{name}\n{eval_results[name]["accuracy"]:.2%}',
                 fontweight='bold')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

plt.tight_layout()
plt.savefig('resultats_complets.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Tous les graphiques sauvegardés dans 'resultats_complets.png' !")

In [ ]:
app_code = '''
import os
import json
import numpy as np
import streamlit as st
from PIL import Image

st.set_page_config(
    page_title="MedAI Classifier",
    page_icon="🏥",
    layout="wide"
)

st.markdown("""
<style>
.main-title {
    font-size: 2rem; font-weight: 700;
    color: #1F4E79; text-align: center;
}
.card {
    background: #f0f4f8; border-radius: 12px;
    padding: 1.2rem; border-left: 4px solid #2E75B6;
    margin-bottom: 1rem;
}
</style>
""", unsafe_allow_html=True)

@st.cache_resource
def get_class_names():
    for path in ["train", "data/train"]:
        if os.path.exists(path):
            cls = sorted([
                f for f in os.listdir(path)
                if os.path.isdir(os.path.join(path, f))
            ])
            if cls:
                return cls
    return ["NORMAL", "PNEUMONIA"]

@st.cache_resource
def load_models():
    from tensorflow.keras.models import load_model
    catalog = {
        "VGG16"      : "best_vgg16.keras",
        "ResNet50"   : "best_resnet50.keras",
        "MobileNetV2": "best_mobilenet.keras"
    }
    loaded = {}
    for name, path in catalog.items():
        if os.path.exists(path):
            try:
                loaded[name] = load_model(path)
                print(f"✅ {name} chargé")
            except Exception as e:
                print(f"❌ {name} : {e}")
    return loaded

def preprocess(img):
    img = img.convert("RGB").resize((224, 224))
    arr = np.array(img, dtype=np.float32) / 255.0
    return np.expand_dims(arr, axis=0)

def predict(model, arr, class_names):
    preds   = model.predict(arr, verbose=0)[0]
    top_idx = int(np.argmax(preds))
    results = sorted(
        zip(class_names, preds.tolist()),
        key=lambda x: x[1], reverse=True
    )
    return class_names[top_idx], float(preds[top_idx]) * 100, results

CLASS_NAMES = get_class_names()
MODELS      = load_models()

st.markdown(\'<h1 class="main-title">🏥 MedAI — Classificateur d\\\'Images Médicales</h1>\',
            unsafe_allow_html=True)
st.markdown("---")

with st.sidebar:
    st.markdown("## ⚙️ Configuration")
    if MODELS:
        model_choice = st.selectbox("Modèle", list(MODELS.keys()))
    else:
        st.error("❌ Aucun modèle trouvé !")
        model_choice = None
    st.markdown("---")
    st.markdown("### Classes")
    for c in CLASS_NAMES:
        st.markdown(f"- **{c}**")

tab1, tab2, tab3 = st.tabs([
    "🔬 Classifier",
    "📊 Performances",
    "📈 Courbes"
])

with tab1:
    col1, col2 = st.columns([1, 1], gap="large")

    with col1:
        st.markdown("### 📤 Image à analyser")
        uploaded = st.file_uploader(
            "Uploader une image",
            type=["jpg", "jpeg", "png"]
        )
        if uploaded:
            img = Image.open(uploaded)
            st.image(img, use_column_width=True)
            c1, c2 = st.columns(2)
            c1.metric("Largeur", f"{img.size[0]} px")
            c2.metric("Hauteur", f"{img.size[1]} px")

    with col2:
        st.markdown("### 🎯 Résultats")
        if uploaded and MODELS and model_choice:
            with st.spinner(f"Analyse avec {model_choice}..."):
                arr = preprocess(img)
                top_class, top_conf, all_results = predict(
                    MODELS[model_choice], arr, CLASS_NAMES
                )
            if top_conf >= 75:
                st.success(f"✅ {top_class} — {top_conf:.1f}%")
            elif top_conf >= 50:
                st.warning(f"⚠️ {top_class} — {top_conf:.1f}%")
            else:
                st.error(f"❌ {top_class} — {top_conf:.1f}%")

            st.markdown("**Probabilités :**")
            for cls, prob in all_results:
                st.progress(int(prob * 100), text=f"{cls} — {prob*100:.1f}%")

            st.markdown("---")
            if top_conf >= 75:
                st.info("✅ Prédiction fiable — peut aider au diagnostic.")
            elif top_conf >= 50:
                st.warning("⚠️ Incertain — analyse complémentaire recommandée.")
            else:
                st.error("❌ Faible confiance — consulter un médecin.")
        else:
            st.info("👆 Uploadez une image pour lancer l\\\'analyse.")

    if uploaded and MODELS and len(MODELS) > 1:
        st.markdown("---")
        st.markdown("### 🏆 Comparaison tous les modèles")
        arr   = preprocess(img)
        cols  = st.columns(len(MODELS))
        best_conf = 0
        best_name = ""
        for name, model in MODELS.items():
            tc, conf, _ = predict(model, arr, CLASS_NAMES)
            if conf > best_conf:
                best_conf = conf
                best_name = name
        for col, (name, model) in zip(cols, MODELS.items()):
            tc, conf, _ = predict(model, arr, CLASS_NAMES)
            with col:
                label = f"🥇 {name}" if name == best_name else name
                st.metric(label, tc, f"{conf:.1f}%")

with tab2:
    st.markdown("### 📊 Résultats d\\\'entraînement")
    import pandas as pd

    histories = {
        "VGG16"      : "history_vgg.json",
        "ResNet50"   : "history_resnet.json",
        "MobileNetV2": "history_mobile.json"
    }

    rows = []
    for name, path in histories.items():
        if os.path.exists(path):
            with open(path) as f:
                h = json.load(f)
            rows.append({
                "Modèle"      : name,
                "Train Acc"   : f"{max(h[\'accuracy\']):.2%}",
                "Val Acc"     : f"{max(h[\'val_accuracy\']):.2%}",
                "Val Loss"    : f"{min(h[\'val_loss\']):.4f}",
                "Epochs"      : len(h[\'accuracy\']),
                "Best Epoch"  : h[\'val_accuracy\'].index(max(h[\'val_accuracy\']))+1
            })

    if rows:
        df = pd.DataFrame(rows)
        st.dataframe(df, use_container_width=True, hide_index=True)
        best = df.loc[df["Val Acc"].idxmax(), "Modèle"]
        st.success(f"🥇 Meilleur modèle : **{best}**")
    else:
        st.info("Lance d\\\'abord l\\\'entraînement pour voir les résultats.")

with tab3:
    st.markdown("### 📈 Courbes d\\\'apprentissage")
    hfiles = {
        "VGG16"      : "history_vgg.json",
        "ResNet50"   : "history_resnet.json",
        "MobileNetV2": "history_mobile.json"
    }
    t1, t2, t3 = st.tabs(list(hfiles.keys()))
    for tab, (name, path) in zip([t1, t2, t3], hfiles.items()):
        with tab:
            if os.path.exists(path):
                with open(path) as f:
                    h = json.load(f)
                import pandas as pd
                c1, c2 = st.columns(2)
                c1.line_chart(pd.DataFrame({
                    "Train"     : h["accuracy"],
                    "Validation": h["val_accuracy"]
                }), height=220)
                c2.line_chart(pd.DataFrame({
                    "Train"     : h["loss"],
                    "Validation": h["val_loss"]
                }), height=220)
                st.success(
                    f"✅ Meilleure Val Accuracy : "
                    f"{max(h[\'val_accuracy\']):.2%}"
                )
            else:
                st.info(f"Entraîne {name} pour voir ses courbes.")

st.markdown("---")
st.markdown(
    "<center><small>MedAI · Deep Learning · Transfer Learning · 2025/2026</small></center>",
    unsafe_allow_html=True
)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✅ app.py créé avec succès !")
print("=" * 40)
print("▶️  Lance dans le terminal :")
print("   streamlit run app.py")
print("=" * 40)

In [ ]:
import os
print("📁 Dossier actuel :", os.getcwd())
print("📋 Fichiers présents :")
for f in os.listdir("."):
    print(f"   {f}")

In [ ]:
app_code = r"""
import os, json
import numpy as np
import streamlit as st
from PIL import Image

st.set_page_config(page_title="MedAI Classifier", page_icon="🏥", layout="wide")

@st.cache_resource
def get_class_names():
    for path in ["train"]:
        if os.path.exists(path):
            cls = sorted([f for f in os.listdir(path) if os.path.isdir(os.path.join(path,f))])
            if cls:
                return cls
    return ["NORMAL", "PNEUMONIA"]

@st.cache_resource
def load_models():
    from tensorflow.keras.models import load_model
    loaded = {}
    for name, path in [("VGG16","best_vgg16.keras"),("ResNet50","best_resnet50.keras"),("MobileNetV2","best_mobilenet.keras")]:
        if os.path.exists(path):
            try:
                loaded[name] = load_model(path)
            except:
                pass
    return loaded

def preprocess(img):
    img = img.convert("RGB").resize((224,224))
    return np.expand_dims(np.array(img,dtype=np.float32)/255.0, axis=0)

def predict(model, arr, class_names):
    preds = model.predict(arr, verbose=0)[0]
    top   = int(np.argmax(preds))
    res   = sorted(zip(class_names, preds.tolist()), key=lambda x: x[1], reverse=True)
    return class_names[top], float(preds[top])*100, res

CLASS_NAMES = get_class_names()
MODELS      = load_models()

st.title("🏥 MedAI — Classificateur d'Images Médicales")
st.caption("Deep Learning · Transfer Learning · VGG16 · ResNet50 · MobileNetV2")
st.markdown("---")

with st.sidebar:
    st.markdown("## ⚙️ Configuration")
    if MODELS:
        model_choice = st.selectbox("Modèle", list(MODELS.keys()))
    else:
        st.error("❌ Aucun modèle trouvé !")
        model_choice = None
    st.markdown("### Classes détectées")
    for c in CLASS_NAMES:
        st.markdown(f"- {c}")

tab1, tab2, tab3 = st.tabs(["🔬 Classifier", "📊 Performances", "📈 Courbes"])

with tab1:
    col1, col2 = st.columns(2, gap="large")
    with col1:
        st.markdown("### 📤 Image à analyser")
        uploaded = st.file_uploader("Uploader une image", type=["jpg","jpeg","png"])
        if uploaded:
            img = Image.open(uploaded)
            st.image(img, use_column_width=True)
            c1, c2 = st.columns(2)
            c1.metric("Largeur", f"{img.size[0]} px")
            c2.metric("Hauteur", f"{img.size[1]} px")

    with col2:
        st.markdown("### 🎯 Résultats")
        if uploaded and MODELS and model_choice:
            with st.spinner(f"Analyse avec {model_choice}..."):
                arr = preprocess(img)
                top_class, top_conf, all_results = predict(MODELS[model_choice], arr, CLASS_NAMES)
            if top_conf >= 75:
                st.success(f"✅ {top_class} — {top_conf:.1f}%")
            elif top_conf >= 50:
                st.warning(f"⚠️ {top_class} — {top_conf:.1f}%")
            else:
                st.error(f"❌ {top_class} — {top_conf:.1f}%")
            st.markdown("**Probabilités :**")
            for cls, prob in all_results:
                st.progress(int(prob*100), text=f"{cls} : {prob*100:.1f}%")
        elif not uploaded:
            st.info("👆 Uploadez une image pour lancer l'analyse.")
        else:
            st.error("❌ Aucun modèle disponible.")

    if uploaded and MODELS and len(MODELS) > 1:
        st.markdown("---")
        st.markdown("### 🏆 Comparaison tous les modèles")
        arr  = preprocess(img)
        cols = st.columns(len(MODELS))
        preds_all = {n: predict(m, arr, CLASS_NAMES) for n,m in MODELS.items()}
        best = max(preds_all, key=lambda x: preds_all[x][1])
        for col, (name, (tc, conf, _)) in zip(cols, preds_all.items()):
            with col:
                st.metric(f"{'🥇 ' if name==best else ''}{name}", tc, f"{conf:.1f}%")

with tab2:
    st.markdown("### 📊 Résultats d'entraînement")
    import pandas as pd
    rows = []
    for name, path in [("VGG16","history_vgg.json"),("ResNet50","history_resnet.json"),("MobileNetV2","history_mobile.json")]:
        if os.path.exists(path):
            with open(path) as f:
                h = json.load(f)
            rows.append({
                "Modèle"    : name,
                "Train Acc" : f"{max(h['accuracy']):.2%}",
                "Val Acc"   : f"{max(h['val_accuracy']):.2%}",
                "Val Loss"  : f"{min(h['val_loss']):.4f}",
                "Epochs"    : len(h['accuracy'])
            })
    if rows:
        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)
    else:
        st.info("Lance d'abord l'entraînement pour voir les résultats.")

with tab3:
    st.markdown("### 📈 Courbes d'apprentissage")
    import pandas as pd
    t1, t2, t3 = st.tabs(["VGG16", "ResNet50", "MobileNetV2"])
    for tab, (name, path) in zip([t1,t2,t3], [("VGG16","history_vgg.json"),("ResNet50","history_resnet.json"),("MobileNetV2","history_mobile.json")]):
        with tab:
            if os.path.exists(path):
                with open(path) as f:
                    h = json.load(f)
                c1, c2 = st.columns(2)
                c1.line_chart(pd.DataFrame({"Train":h["accuracy"],"Validation":h["val_accuracy"]}), height=220)
                c2.line_chart(pd.DataFrame({"Train":h["loss"],"Validation":h["val_loss"]}), height=220)
                st.success(f"✅ Meilleure Val Accuracy : {max(h['val_accuracy']):.2%}")
            else:
                st.info(f"Entraîne {name} pour voir ses courbes.")

st.markdown("---")
st.caption("MedAI · Deep Learning · Transfer Learning · 2025/2026")
"""

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("✅ app.py créé dans :", os.getcwd())
print("▶️  Lance : streamlit run app.py")

In [ ]:
import json

for name, path in [("VGG16","history_vgg.json"),
                   ("ResNet50","history_resnet.json"),
                   ("MobileNetV2","history_mobile.json")]:
    with open(path) as f:
        h = json.load(f)
    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    for i, (acc, val) in enumerate(zip(h['accuracy'], h['val_accuracy'])):
        trend = "📈" if i > 0 and val > h['val_accuracy'][i-1] else "📉"
        print(f"  Epoch {i+1} | Train: {acc:.2%} | Val: {val:.2%} {trend}")
    print(f"  Meilleure Val Acc : {max(h['val_accuracy']):.2%}")
    print(f"  Dernière Val Acc  : {h['val_accuracy'][-1]:.2%}")